# Реализация и проектирование реляционных баз данных
## Итоговый проект. Часть 2

```
Контрольная точка: #2
Группа: 2ЭИТ1-9.24
Выполнил: Бирюков Николай
Тема: Yandex Music — датасет Yambda
```

---

### О проекте

Практическая реализация реляционной базы данных на основе даталогической модели,
разработанной в ходе первой контрольной точки (датасет **Yambda** — Yandex Music Benchmark Dataset).

**СУБД:**
- **PostgreSQL** + расширение **pgvector** (`vector(128)`, HNSW-индекс)
- **MariaDB 11.7+** с нативной поддержкой векторов (`VECTOR(128)`, `VECTOR INDEX`)

**Даталогическая модель (8 таблиц):**

| Таблица | PK | FK | Примечание |
|---------|----|----|------------|
| `event_type` | `event_type_id` | — | Справочник: 5 значений |
| `usr` | `user_id` | — | uid из датасета |
| `artist` | `artist_id` | — | |
| `album` | `album_id` | — | |
| `track` | `track_id` | — | `embed VECTOR(128)` |
| `artist_track` | `(artist_id, track_id)` | artist, track | M:M |
| `album_track` | `(album_id, track_id)` | album, track | M:M |
| `event` | `event_id` | usr, track, event_type | Таблица фактов |

**Источники данных** (`var/data/`):
`album_item_mapping.parquet`, `artist_item_mapping.parquet`, `embeddings.parquet`,
`listens.parquet`, `likes.parquet`, `dislikes.parquet`, `unlikes.parquet`,
`undislikes.parquet`, `multi_event.parquet`

### Об исходном датасете Yambda

**Yambda-5B** (Yandex Music Benchmark Dataset) — промышленно-масштабный открытый датасет
музыкальных взаимодействий, опубликованный Яндексом для научных и исследовательских целей
(лицензия **Apache 2.0**, препринт [arXiv:2505.22238](https://arxiv.org/abs/2505.22238)).
Содержит как явную обратную связь (лайки/дизлайки), так и неявную (прослушивания), а также
маркер органических vs рекомендательных взаимодействий и предобученные аудио-эмбеддинги.

**Объём:** 4.79 млрд взаимодействий, 1 млн пользователей, 9.39 млн треков;
аудио-эмбеддинги для 7.72 млн треков (128-мерные, получены CNN в парадигме Contrastive
Learning of Musical Representations, Spijkervet et al., 2021).

Датасет выпускается в трёх масштабах и двух форматах:

| Масштаб | Пользователи | Треки | Прослушивания | Лайки | Дизлайки |
|---------|--------------|-------|---------------|-------|----------|
| Yambda-50M | 10 000 | 934 057 | 46 467 212 | 881 456 | 107 776 |
| Yambda-500M | 100 000 | 3 004 578 | 466 512 103 | 9 033 960 | 1 128 113 |
| Yambda-5B | 1 000 000 | 9 390 623 | 4 649 567 411 | 89 334 605 | 11 579 143 |

- **`flat`** — по одному файлу на тип события (гомогенный формат), сортировка по `(uid, timestamp)`.
- **`sequential`** — агрегированный по пользователям формат (списки `item_ids`/`timestamps`/...);
  удобен для моделей последовательных рекомендаций.

В данном проекте используется срез **`sequential/500m`** (100k пользователей, ~480M строк в
унифицированном `multi_event.parquet`).

**Состав файлов и схемы:**

| Файл | Описание | Схема |
|------|----------|-------|
| `listens.parquet` | События прослушивания | `uid, item_id, timestamp, is_organic, played_ratio_pct, track_length_seconds` |
| `likes.parquet` | Лайки | `uid, item_id, timestamp, is_organic` |
| `dislikes.parquet` | Дизлайки | `uid, item_id, timestamp, is_organic` |
| `unlikes.parquet` | Отмена лайка | `uid, item_id, timestamp, is_organic` |
| `undislikes.parquet` | Отмена дизлайка | `uid, item_id, timestamp, is_organic` |
| `multi_event.parquet` | Унифицированные события | `uid, item_id, timestamp, is_organic, event_type, played_ratio_pct, track_length_seconds` |
| `embeddings.parquet` | Аудио-эмбеддинги треков | `item_id, embed, normalized_embed` |

**Поля событий:**
- `uid` (uint32) — идентификатор пользователя;
- `item_id` (uint32) — идентификатор трека;
- `timestamp` (uint32) — время, сгруппированное в корзины по 5 секунд (delta times);
- `is_organic` (uint8) — `1` для органических взаимодействий, `0` для рекомендательных
  (персональный фид / персональные плейлисты);
- `event_type` (enum) — `listen | like | dislike | unlike | undislike`;
- `played_ratio_pct` (uint16, nullable) — доля прослушанного, **1–100 для `listen`, null иначе**;
- `track_length_seconds` (uint32, nullable) — длительность трека, null для не-`listen`.

> **Важно для схемы.** Согласно [FAQ датасета](https://huggingface.co/datasets/yandex/yambda/discussions/10),
> значения `played_ratio_pct > 100` — **ожидаемое поведение**, а не шум: они возникают, когда
> пользователь перематывал и переслушивал фрагменты трека, так что суммарное время
> прослушивания превышает длительность трека. Это учитывается при проектировании ограничений
> на столбец `event.played_ratio_pct` (CHECK не должен отсекать значения `> 100`).

**Ключевые особенности датасета:**
- глобальное временное упорядочивание событий;
- маркер органических vs рекомендательных взаимодействий;
- 128-мерные аудио-эмбеддинги для content-aware рекомендаций;
- стандартизированный протокол оценки и базовые бенчмарки (`benchmarks/`).

---
## Зависимости и конфигурация

In [ ]:
!pip install pandas duckdb psycopg2-binary pgvector pymysql python-dotenv tqdm --quiet

In [ ]:
import pandas as pd
import duckdb
import os
import psycopg2
import pymysql
import numpy as np
from pgvector.psycopg2 import register_vector
from dotenv import load_dotenv

load_dotenv()

DATA_DIR = 'datasets/yambda'
EVENTS_DIR = 'sequential/500m'
PARQUET = {
    'album_item_mapping' : f'{DATA_DIR}/album_item_mapping.parquet',
    'artist_item_mapping': f'{DATA_DIR}/artist_item_mapping.parquet',
    'embeddings'         : f'{DATA_DIR}/embeddings.parquet',
    'listens'            : f'{DATA_DIR}/{EVENTS_DIR}/listens.parquet',
    'likes'              : f'{DATA_DIR}/{EVENTS_DIR}/likes.parquet',
    'dislikes'           : f'{DATA_DIR}/{EVENTS_DIR}/dislikes.parquet',
    'unlikes'            : f'{DATA_DIR}/{EVENTS_DIR}/unlikes.parquet',
    'undislikes'         : f'{DATA_DIR}/{EVENTS_DIR}/undislikes.parquet',
    'multi_event'        : f'{DATA_DIR}/{EVENTS_DIR}/multi_event.parquet',
}

pg_user = os.getenv('PG_CONN_USER')
print(f'Postgres user: {pg_user}')

PG_CONFIG = {
    'host': '192.168.200.128', 'port': 5432,
    'dbname': 'yambda2', 'user': os.getenv('PG_CONN_USER'), 'password': os.getenv('PG_CONN_PWD'),
}
MARIADB_CONFIG = {
    'host': '192.168.200.128', 'port': 3306,
    'database': 'yambda', 'user': os.getenv('PG_CONN_USER'), 'password': os.getenv('PG_CONN_PWD'),
    'local_infile': True, 'read_timeout': 7200, 'write_timeout': 7200,
}

# Ограничение строк на тип события при загрузке (None = все ~500M)
EVENT_LIMIT = None  # None = полная загрузка ~480M; 100_000 для теста

# True → выполнять ячейки загрузки данных 16–20 
ENABLE_DATA_LOAD = False  # данные уже загружены в yambda2 (PG) и yambda (MA)

print('Конфигурация загружена.')

---
## Этапы 1–3 (PostgreSQL): Установка и настройка сервера

**Критерий (+1):** Продемонстрированы все скриншоты обозревателя объектов.  
> **TBD:** Скриншоты pgAdmin после заполнения `PG_CONFIG`.

Подключение, включение `pgvector`, проверка версий.

In [ ]:
pg_conn = psycopg2.connect(**PG_CONFIG)
pg_conn.autocommit = True
pg_cur = pg_conn.cursor()

pg_cur.execute('SELECT version();')
print('PostgreSQL:', pg_cur.fetchone()[0])

# Проверяем наличие pgvector (установлен заранее на сервере)
pg_cur.execute("SELECT extversion FROM pg_extension WHERE extname = 'vector';")
row = pg_cur.fetchone()
if row:
    print('pgvector версия:', row[0])
    register_vector(pg_conn)
    print('register_vector: OK')
else:
    raise RuntimeError('pgvector не найден в базе yambda — векторный поиск невозможен')

---
## Этапы 22–23 (MariaDB): Установка и настройка сервера

**Критерий (+1):** Продемонстрированы все скриншоты создания сервера.  
> **TBD:** Скриншоты DBeaver / HeidiSQL после заполнения `MARIADB_CONFIG`.

MariaDB 11.7+ — векторная поддержка встроена нативно, расширения не требуются.

In [ ]:
maria_conn = pymysql.connect(**MARIADB_CONFIG)
maria_cur = maria_conn.cursor()
maria_cur.execute('SELECT VERSION();')
print('MariaDB:', maria_cur.fetchone()[0])

---
## Этапы 4–5 (PostgreSQL): Создание таблиц — DDL

**Критерии:**
- Реализованы все таблицы по даталогической модели и словарю данных (+7)
- Продемонстрированы таблицы, столбцы и ограничения (+1)
- Количество таблиц совпадает с моделью (+1)
- Количество столбцов совпадает с моделью (+1)

In [ ]:
PG_DDL = '''
CREATE TABLE IF NOT EXISTS event_type (
    event_type_id   SERIAL      NOT NULL CONSTRAINT PK_event_type PRIMARY KEY,
    event_type_name VARCHAR(20) NOT NULL CONSTRAINT UQ_event_type_name UNIQUE
                                         CONSTRAINT CH_event_type_name
                                         CHECK (event_type_name IN
                                         (\'listen\',\'like\',\'dislike\',\'unlike\',\'undislike\'))
);
CREATE TABLE IF NOT EXISTS usr (
    user_id BIGINT NOT NULL CONSTRAINT PK_usr PRIMARY KEY
);
CREATE TABLE IF NOT EXISTS artist (
    artist_id INTEGER NOT NULL CONSTRAINT PK_artist PRIMARY KEY
);
CREATE TABLE IF NOT EXISTS album (
    album_id INTEGER NOT NULL CONSTRAINT PK_album PRIMARY KEY
);
CREATE TABLE IF NOT EXISTS track (
    track_id             INTEGER     NOT NULL CONSTRAINT PK_track PRIMARY KEY,
    track_length_seconds INTEGER     NULL,
    embed                VECTOR(128) NULL,
    normalized_embed     VECTOR(128) NULL
);
CREATE INDEX IF NOT EXISTS IDX_track_embed
    ON track USING hnsw (embed vector_cosine_ops);
CREATE INDEX IF NOT EXISTS IDX_track_norm_embed
    ON track USING hnsw (normalized_embed vector_cosine_ops);
CREATE TABLE IF NOT EXISTS artist_track (
    artist_id INTEGER NOT NULL REFERENCES artist (artist_id),
    track_id  INTEGER NOT NULL REFERENCES track  (track_id),
    CONSTRAINT PK_artist_track PRIMARY KEY (artist_id, track_id)
);
CREATE TABLE IF NOT EXISTS album_track (
    album_id INTEGER NOT NULL REFERENCES album (album_id),
    track_id INTEGER NOT NULL REFERENCES track (track_id),
    CONSTRAINT PK_album_track PRIMARY KEY (album_id, track_id)
);
CREATE TABLE IF NOT EXISTS event (
    event_id         BIGSERIAL NOT NULL CONSTRAINT PK_event PRIMARY KEY,
    user_id          BIGINT    NOT NULL REFERENCES usr        (user_id),
    track_id         INTEGER   NOT NULL REFERENCES track      (track_id),
    event_type_id    INTEGER   NOT NULL REFERENCES event_type (event_type_id),
    ts               BIGINT    NOT NULL,
    is_organic       SMALLINT  NOT NULL CONSTRAINT CH_is_organic
                                         CHECK (is_organic IN (0, 1)),
    -- played_ratio_pct > 100 валиден (перемотка) — CHECK на диапазон не ставим
    played_ratio_pct REAL      NULL
);
'''

pg_cur.execute(PG_DDL)
print('PostgreSQL: все таблицы созданы.')

In [ ]:
# Проверка количества таблиц и столбцов (PostgreSQL)
pg_cur.execute('''
    SELECT COUNT(*) FROM information_schema.tables
    WHERE table_schema = \'public\' AND table_type = \'BASE TABLE\';
''')
print('Таблиц (PostgreSQL):', pg_cur.fetchone()[0])

pg_cur.execute('''
    SELECT table_name, COUNT(column_name) AS cols
    FROM information_schema.columns
    WHERE table_schema = \'public\'
    GROUP BY table_name ORDER BY table_name;
''')
for row in pg_cur.fetchall():
    print(f'  {row[0]}: {row[1]} столбцов')

---
## Этапы 24–25 (MariaDB): Создание таблиц — DDL

**Критерии:** те же, что в этапах 4–5, для MariaDB.

Отличия от PostgreSQL: `AUTO_INCREMENT`, нативный `VECTOR(128)`, `VECTOR INDEX`, `TINYINT`.

In [ ]:
MARIA_DDL = [
    '''CREATE TABLE IF NOT EXISTS event_type (
        event_type_id   INT         NOT NULL AUTO_INCREMENT PRIMARY KEY,
        event_type_name VARCHAR(20) NOT NULL UNIQUE
            CHECK (event_type_name IN ('listen','like','dislike','unlike','undislike'))
    );''',
    'CREATE TABLE IF NOT EXISTS usr   (user_id   BIGINT NOT NULL PRIMARY KEY);',
    'CREATE TABLE IF NOT EXISTS artist(artist_id INT    NOT NULL PRIMARY KEY);',
    'CREATE TABLE IF NOT EXISTS album (album_id  INT    NOT NULL PRIMARY KEY);',
    # VECTOR INDEX требует NOT NULL-столбцы — индекс добавляется после загрузки и zero-fill, не в DDL
    '''CREATE TABLE IF NOT EXISTS track (
        track_id             INT         NOT NULL PRIMARY KEY,
        track_length_seconds INT         NULL,
        embed                VECTOR(128) NULL,
        normalized_embed     VECTOR(128) NULL
    );''',
    '''CREATE TABLE IF NOT EXISTS artist_track (
        artist_id INT NOT NULL REFERENCES artist (artist_id),
        track_id  INT NOT NULL REFERENCES track  (track_id),
        PRIMARY KEY (artist_id, track_id)
    );''',
    '''CREATE TABLE IF NOT EXISTS album_track (
        album_id INT NOT NULL REFERENCES album (album_id),
        track_id INT NOT NULL REFERENCES track (track_id),
        PRIMARY KEY (album_id, track_id)
    );''',
    # played_ratio_pct > 100 валиден (перемотка) — CHECK на диапазон не ставим
    '''CREATE TABLE IF NOT EXISTS event (
        event_id         BIGINT  NOT NULL AUTO_INCREMENT PRIMARY KEY,
        user_id          BIGINT  NOT NULL REFERENCES usr        (user_id),
        track_id         INT     NOT NULL REFERENCES track      (track_id),
        event_type_id    INT     NOT NULL REFERENCES event_type (event_type_id),
        ts               BIGINT  NOT NULL,
        is_organic       TINYINT NOT NULL CHECK (is_organic IN (0,1)),
        played_ratio_pct FLOAT   NULL
    );''',
]

for stmt in MARIA_DDL:
    maria_cur.execute(stmt)
maria_conn.commit()
print('MariaDB: все таблицы созданы.')

In [ ]:
maria_cur.execute('''
    SELECT COUNT(*) FROM information_schema.tables
    WHERE table_schema = DATABASE() AND table_type = \'BASE TABLE\';
''')
print('Таблиц (MariaDB):', maria_cur.fetchone()[0])

maria_cur.execute('''
    SELECT table_name, COUNT(column_name) AS cols
    FROM information_schema.columns
    WHERE table_schema = DATABASE()
    GROUP BY table_name ORDER BY table_name;
''')
for row in maria_cur.fetchall():
    print(f'  {row[0]}: {row[1]} столбцов')

---
## Этапы 6, 28 (PG + MariaDB): Загрузка данных — DML INSERT

**Критерии:**
- Каждая таблица заполнена данными в соответствии с 3НФ (+8 для каждой СУБД)
- Запросы SELECT не выводят суррогатные/внешние ключи (-1 за каждый)
- Количество JOIN = количеству FK (-1 если меньше)

### Методология

| Параметр | PostgreSQL | MariaDB |
|----------|-----------|---------|
| Экспорт | DuckDB → PSV (streaming) | DuckDB → PSV |
| Загрузка | COPY FROM STDIN (12 потоков) | LOAD DATA LOCAL INFILE (однопоточно) |
| Скорость | ~500k строк/сек/поток | ~400k строк/сек |

Параллельная загрузка в MariaDB не работает — deadlock 1213 на index contention.
PostgreSQL справляется с 12 параллельными COPY.

Порядок по FK: event_type → artist → album → track → usr → artist_track → album_track → event
Векторы — отдельным шагом через staging-таблицу.

*Ячейки загрузки 16–20 пропускаются при `ENABLE_DATA_LOAD = False` (см. конфигурацию).*


In [ ]:
if ENABLE_DATA_LOAD:
    # event_type (hardcoded — нет parquet-источника)
    ET = [(1,'listen'),(2,'like'),(3,'dislike'),(4,'unlike'),(5,'undislike')]
    pg_cur.executemany(
        'INSERT INTO event_type(event_type_id,event_type_name) VALUES(%s,%s) ON CONFLICT DO NOTHING', ET)
    maria_cur.executemany(
        'INSERT IGNORE INTO event_type(event_type_id,event_type_name) VALUES(%s,%s)', ET)
    maria_conn.commit()

    # artist
    artists = duckdb.sql(
        f"SELECT DISTINCT artist_id FROM '{PARQUET['artist_item_mapping']}' ORDER BY 1"
    ).df()['artist_id'].tolist()
    pg_cur.executemany('INSERT INTO artist(artist_id) VALUES(%s) ON CONFLICT DO NOTHING',
                       [(x,) for x in artists])
    maria_cur.executemany('INSERT IGNORE INTO artist(artist_id) VALUES(%s)',
                          [(x,) for x in artists])
    maria_conn.commit()

    # album
    albums = duckdb.sql(
        f"SELECT DISTINCT album_id FROM '{PARQUET['album_item_mapping']}' ORDER BY 1"
    ).df()['album_id'].tolist()
    pg_cur.executemany('INSERT INTO album(album_id) VALUES(%s) ON CONFLICT DO NOTHING',
                       [(x,) for x in albums])
    maria_cur.executemany('INSERT IGNORE INTO album(album_id) VALUES(%s)',
                          [(x,) for x in albums])
    maria_conn.commit()

    print(f'event_type: 5 | artist: {len(artists):,} | album: {len(albums):,}')
else:
    print('⏭ Skip: ENABLE_DATA_LOAD = False — данные уже загружены')


In [ ]:
if ENABLE_DATA_LOAD:
    import os, time

    WORK = os.path.expanduser('~/nikolyabb/_load')
    os.makedirs(WORK, exist_ok=True)
    NIL = chr(92) + 'N'

    track_psv = f'{WORK}/track.psv'
    duckdb.sql(f"""
        COPY (
            WITH tracks AS (
                SELECT item_id FROM '{PARQUET['embeddings']}'
                UNION
                SELECT DISTINCT item_id[1] AS item_id FROM '{PARQUET['listens']}'
                UNION SELECT DISTINCT item_id[1] FROM '{PARQUET['likes']}'
                UNION SELECT DISTINCT item_id[1] FROM '{PARQUET['dislikes']}'
                UNION SELECT DISTINCT item_id[1] FROM '{PARQUET['unlikes']}'
                UNION SELECT DISTINCT item_id[1] FROM '{PARQUET['undislikes']}'
            ),
            lengths AS (
                SELECT DISTINCT item_id[1] AS item_id, track_length_seconds[1] AS track_length_seconds
                FROM '{PARQUET['multi_event']}'
            )
            SELECT t.item_id AS track_id, l.track_length_seconds
            FROM tracks t LEFT JOIN lengths l ON t.item_id = l.item_id
            ORDER BY t.item_id
        ) TO '{track_psv}' (HEADER false, DELIM '|', NULLSTR '{NIL}')
    """)

    # PostgreSQL
    t0 = time.time()
    pg_cur.execute('SET synchronous_commit=off')
    with open(track_psv, 'rb') as f:
        pg_cur.copy_expert(
            f"COPY track(track_id, track_length_seconds) FROM STDIN WITH (FORMAT text, DELIMITER '|', NULL '{NIL}')", f)
    pg_conn.commit()
    pg_cur.execute("SELECT count(*) FROM track")
    print(f'PG track: {pg_cur.fetchone()[0]:,} в {time.time()-t0:.1f}s')

    # MariaDB
    t0 = time.time()
    maria_cur.execute('SET foreign_key_checks=0')
    maria_cur.execute(f"LOAD DATA LOCAL INFILE '{track_psv}' INTO TABLE track FIELDS TERMINATED BY '|' (track_id, track_length_seconds)")
    maria_conn.commit()
    maria_cur.execute("SELECT count(*) FROM track")
    print(f'MariaDB track: {maria_cur.fetchone()[0]:,} в {time.time()-t0:.1f}s')
else:
    print('⏭ Skip: ENABLE_DATA_LOAD = False — данные уже загружены')


In [ ]:
if ENABLE_DATA_LOAD:
    # artist_track
    at_psv = f'{WORK}/artist_track.psv'
    duckdb.sql(f"COPY (SELECT artist_id, item_id AS track_id FROM '{PARQUET['artist_item_mapping']}') TO '{at_psv}' (HEADER false, DELIM '|')")

    t0 = time.time()
    with open(at_psv, 'rb') as f:
        pg_cur.copy_expert(f"COPY artist_track(artist_id, track_id) FROM STDIN WITH (FORMAT text, DELIMITER '|', NULL '{NIL}')", f)
    pg_conn.commit()
    maria_cur.execute(f"LOAD DATA LOCAL INFILE '{at_psv}' INTO TABLE artist_track FIELDS TERMINATED BY '|' (artist_id, track_id)")
    maria_conn.commit()
    pg_cur.execute("SELECT count(*) FROM artist_track"); pg_c = pg_cur.fetchone()[0]
    maria_cur.execute("SELECT count(*) FROM artist_track"); ma_c = maria_cur.fetchone()[0]
    print(f'artist_track: PG={pg_c:,} MA={ma_c:,} в {time.time()-t0:.1f}s')
    os.remove(at_psv)

    # album_track
    abt_psv = f'{WORK}/album_track.psv'
    duckdb.sql(f"COPY (SELECT album_id, item_id AS track_id FROM '{PARQUET['album_item_mapping']}') TO '{abt_psv}' (HEADER false, DELIM '|')")

    t0 = time.time()
    with open(abt_psv, 'rb') as f:
        pg_cur.copy_expert(f"COPY album_track(album_id, track_id) FROM STDIN WITH (FORMAT text, DELIMITER '|', NULL '{NIL}')", f)
    pg_conn.commit()
    maria_cur.execute(f"LOAD DATA LOCAL INFILE '{abt_psv}' INTO TABLE album_track FIELDS TERMINATED BY '|' (album_id, track_id)")
    maria_conn.commit()
    pg_cur.execute("SELECT count(*) FROM album_track"); pg_c = pg_cur.fetchone()[0]
    maria_cur.execute("SELECT count(*) FROM album_track"); ma_c = maria_cur.fetchone()[0]
    print(f'album_track: PG={pg_c:,} MA={ma_c:,} в {time.time()-t0:.1f}s')
    os.remove(abt_psv)
else:
    print('⏭ Skip: ENABLE_DATA_LOAD = False — данные уже загружены')


In [ ]:
if ENABLE_DATA_LOAD:
    import subprocess, time
    from concurrent.futures import ThreadPoolExecutor, as_completed
    import psycopg2

    # Пользователи — извлекаются из event-файлов
    all_uids = set()
    LIM = f'LIMIT {EVENT_LIMIT}' if EVENT_LIMIT else ''
    for name in ['listens','likes','dislikes','unlikes','undislikes']:
        uids = duckdb.sql(f"SELECT DISTINCT uid FROM '{PARQUET[name]}' {LIM}").df()['uid'].tolist()
        all_uids.update(uids)

    user_rows = [(uid,) for uid in all_uids]
    pg_cur.executemany('INSERT INTO usr(user_id) VALUES(%s) ON CONFLICT DO NOTHING', user_rows)
    maria_cur.executemany('INSERT IGNORE INTO usr(user_id) VALUES(%s)', user_rows)
    maria_conn.commit()
    print(f'usr: {len(user_rows):,}')

    # События — 480M строк, массовая загрузка
    # 1. DuckDB streaming экспорт 6 event-файлов → event.psv
    ET = {'listens':1, 'likes':2, 'dislikes':3, 'unlikes':4, 'undislikes':5}
    parts = []
    for name in ['listens','likes','dislikes','unlikes','undislikes']:
        ratio = 'played_ratio_pct' if name == 'listens' else 'NULL AS played_ratio_pct'
        parts.append(
            f"SELECT uid AS user_id, item_id[1] AS track_id, {ET[name]} AS event_type_id, "
            f"timestamp AS ts, is_organic, {ratio} FROM '{PARQUET[name]}' {LIM}")

    event_psv = f'{WORK}/event.psv'
    duckdb.sql(f"COPY ({' UNION ALL '.join(parts)}) TO '{event_psv}' (HEADER false, DELIM '|', NULLSTR '{NIL}')")

    # 2. PostgreSQL: 12 параллельных COPY
    N = 12
    subprocess.run(['split', '-n', f'l/{N}', event_psv, f'{event_psv}.part_'], check=True)
    split_parts = sorted(f'{WORK}/{f}' for f in os.listdir(WORK) if f.startswith('event.psv.part_'))

    def pg_copy_part(fp):
        c = psycopg2.connect(**PG_CONFIG)
        c.autocommit = False
        cur = c.cursor()
        cur.execute('SET synchronous_commit=off')
        with open(fp, 'rb') as fh:
            cur.copy_expert(
                f"COPY event(user_id,track_id,event_type_id,ts,is_organic,played_ratio_pct) "
                f"FROM STDIN WITH (FORMAT text, DELIMITER '|', NULL '{NIL}')", fh)
        c.commit(); cur.close(); c.close()
        return fp

    t0 = time.time()
    with ThreadPoolExecutor(max_workers=N) as ex:
        for fut in as_completed([ex.submit(pg_copy_part, p) for p in split_parts]):
            os.remove(fut.result())
    pg_cur.execute("SELECT count(*) FROM event")
    print(f'PG event: {pg_cur.fetchone()[0]:,} в {time.time()-t0:.1f}s')

    # 3. MariaDB: LOAD DATA (однопоточно — параллельная вызывает deadlock 1213)
    t0 = time.time()
    maria_cur.execute(f"LOAD DATA LOCAL INFILE '{event_psv}' INTO TABLE event "
                      f"FIELDS TERMINATED BY '|' (user_id, track_id, event_type_id, ts, is_organic, played_ratio_pct)")
    maria_conn.commit()
    maria_cur.execute("SELECT count(*) FROM event")
    print(f'MariaDB event: {maria_cur.fetchone()[0]:,} в {time.time()-t0:.1f}s')

    # Очистка
    for f in os.listdir(WORK):
        if f.startswith('event.psv.part_'): os.remove(f'{WORK}/{f}')
else:
    print('⏭ Skip: ENABLE_DATA_LOAD = False — данные уже загружены')


In [ ]:
if ENABLE_DATA_LOAD:
    # Векторы (embeddings) — загрузка через staging-таблицу
    # MariaDB требует VEC_FromText() — прямой INSERT строкой не работает
    # PG: COPY в temp staging → UPDATE track
    # MariaDB: LOAD DATA + SET col=VEC_FromText(@var) → UPDATE track JOIN _vec_stage

    vec_psv = f'{WORK}/vectors.psv'
    duckdb.sql(f"COPY (SELECT item_id AS track_id, embed::VARCHAR, normalized_embed::VARCHAR "
              f"FROM '{PARQUET['embeddings']}') TO '{vec_psv}' (HEADER false, DELIM '|', NULLSTR '{NIL}')")

    # PostgreSQL
    t0 = time.time()
    pg_cur.execute('SET synchronous_commit=off')
    pg_cur.execute('CREATE TEMP TABLE _vec_stage (track_id int, embed vector(128), normalized_embed vector(128))')
    with open(vec_psv, 'rb') as f:
        pg_cur.copy_expert(
            f"COPY _vec_stage(track_id, embed, normalized_embed) FROM STDIN "
            f"WITH (FORMAT text, DELIMITER '|', NULL '{NIL}')", f)
    pg_cur.execute("UPDATE track SET embed=s.embed, normalized_embed=s.normalized_embed "
                   "FROM _vec_stage s WHERE track.track_id=s.track_id")
    pg_conn.commit()
    pg_cur.execute("SELECT count(*) FROM track WHERE embed IS NOT NULL")
    print(f'PG vectors: {pg_cur.fetchone()[0]:,} в {time.time()-t0:.1f}s')

    # MariaDB — VEC_FromText нужен для парсинга JSON-массива в binary vector
    t0 = time.time()
    maria_cur.execute('CREATE TABLE IF NOT EXISTS _vec_stage (track_id int, embed vector(128), normalized_embed vector(128))')
    maria_cur.execute('TRUNCATE TABLE _vec_stage')
    maria_cur.execute(f"LOAD DATA LOCAL INFILE '{vec_psv}' INTO TABLE _vec_stage "
                      f"FIELDS TERMINATED BY '|' (track_id, @embed, @norm_embed) "
                      f"SET embed=VEC_FromText(@embed), normalized_embed=VEC_FromText(@norm_embed)")
    maria_cur.execute("UPDATE track t INNER JOIN _vec_stage s ON t.track_id=s.track_id "
                      "SET t.embed=s.embed, t.normalized_embed=s.normalized_embed")
    maria_conn.commit()
    maria_cur.execute("SELECT count(*) FROM track WHERE embed IS NOT NULL")
    print(f'MariaDB vectors: {maria_cur.fetchone()[0]:,} в {time.time()-t0:.1f}s')
    maria_cur.execute('DROP TABLE _vec_stage')
    maria_conn.commit()
    os.remove(vec_psv)
else:
    print('⏭ Skip: ENABLE_DATA_LOAD = False — данные уже загружены')


In [ ]:
# Проверка: count(*) для всех таблиц против целевых значений
TARGETS = {
    'event_type': 5, 'usr': 100000, 'artist': 1293394, 'album': 3367691,
    'track': 8088303, 'artist_track': 9271906, 'album_track': 9651644, 'event': 480255564,
}

print(f'{"Таблица":<16} {"Target":>12} {"PG":>12} {"MA":>12}')
print('-' * 58)
for tbl, target in TARGETS.items():
    pg_cur.execute(f"SELECT count(*) FROM {tbl}")
    pg_cnt = pg_cur.fetchone()[0]
    maria_cur.execute(f"SELECT count(*) FROM {tbl}")
    ma_cnt = maria_cur.fetchone()[0]
    ok = ' OK' if pg_cnt == target and ma_cnt == target else ' MISMATCH'
    print(f'{tbl:<16} {target:>12,} {pg_cnt:>12,} {ma_cnt:>12,}{ok}')


---
## Этапы 6, 28 (PG + MariaDB): Запросы SELECT

**Критерии:**
- Запросы адекватны предметной области (+8)
- Нет суррогатных и FK-столбцов в выводе (-1 за каждый)
- JOIN-ов не меньше числа FK в таблице (-1 если меньше)

Вспомогательная функция `pg_q` используется для обоих блоков.

In [ ]:
def pg_q(sql, label='', params=None):
    pg_cur.execute(sql, params)
    cols = [d[0] for d in pg_cur.description]
    df = pd.DataFrame(pg_cur.fetchall(), columns=cols)
    if label: print(f'\n=== {label} ===')
    display(df)
    return df

def maria_q(sql, label=''):
    maria_cur.execute(sql)
    cols = [d[0] for d in maria_cur.description]
    df = pd.DataFrame(maria_cur.fetchall(), columns=cols)
    if label: print(f'\n=== {label} ===')
    display(df)
    return df

In [ ]:
# SELECT 1: Справочник типов событий (event_type — нет FK)
pg_q('''
    SELECT event_type_name AS "Тип события"
    FROM event_type ORDER BY event_type_id;
''', 'Справочник типов событий (PostgreSQL)')

# SELECT 2: Треки с несколькими артистами (artist_track: 2 FK -> 2 JOIN)
pg_q('''
    SELECT t.track_length_seconds                       AS "Длина (сек)",
           COUNT(DISTINCT a.artist_id)                  AS "Число артистов",
           STRING_AGG(DISTINCT al.album_id::text, \', \') AS "Альбомы (ID)"
    FROM track t
    JOIN artist_track att ON t.track_id  = att.track_id
    JOIN artist a         ON att.artist_id = a.artist_id
    LEFT JOIN album_track abt ON t.track_id = abt.track_id
    LEFT JOIN album al        ON abt.album_id = al.album_id
    GROUP BY t.track_id, t.track_length_seconds
    HAVING COUNT(DISTINCT a.artist_id) > 1
    ORDER BY COUNT(DISTINCT a.artist_id) DESC
    LIMIT 10;
''', 'Треки с несколькими артистами (PostgreSQL)')

# SELECT 3: Статистика событий по типу (event: 3 FK -> 3 JOIN)
pg_q('''
    SELECT et.event_type_name                          AS "Тип события",
           COUNT(*)                                    AS "Количество событий",
           ROUND(AVG(e.played_ratio_pct)::numeric, 2)  AS "Среднее прослушивание (%)"
    FROM event e
    JOIN event_type et ON e.event_type_id = et.event_type_id
    JOIN usr u         ON e.user_id       = u.user_id
    JOIN track t       ON e.track_id      = t.track_id
    GROUP BY et.event_type_name
    ORDER BY COUNT(*) DESC;
''', 'Статистика событий по типу (PostgreSQL)')

# SELECT 4: Топ-10 активных пользователей (event: 3 FK -> 3 JOIN)
pg_q('''
    SELECT COUNT(*)                                                    AS "Всего событий",
           SUM(CASE WHEN et.event_type_name = \'listen\' THEN 1 ELSE 0 END) AS "Прослушиваний",
           SUM(CASE WHEN et.event_type_name = \'like\'   THEN 1 ELSE 0 END) AS "Лайков",
           ROUND(AVG(e.played_ratio_pct)::numeric, 1)                  AS "Среднее прослушивание (%)"
    FROM event e
    JOIN usr u         ON e.user_id       = u.user_id
    JOIN event_type et ON e.event_type_id = et.event_type_id
    JOIN track t       ON e.track_id      = t.track_id
    GROUP BY u.user_id
    ORDER BY COUNT(*) DESC
    LIMIT 10;
''', 'Топ-10 активных пользователей (PostgreSQL)')

# SELECT 5: Распределение длин треков (track — нет FK)
pg_q('''
    SELECT CASE
               WHEN track_length_seconds < 120  THEN \'< 2 мин\'
               WHEN track_length_seconds < 240  THEN \'2–4 мин\'
               WHEN track_length_seconds < 360  THEN \'4–6 мин\'
               ELSE                                  \'>= 6 мин\'
           END                            AS "Категория длины",
           COUNT(*)                       AS "Число треков",
           ROUND(AVG(track_length_seconds)::numeric, 0) AS "Средняя длина (сек)"
    FROM track
    WHERE track_length_seconds IS NOT NULL
    GROUP BY 1 ORDER BY MIN(track_length_seconds);
''', 'Распределение треков по длине (PostgreSQL)')

---
## Этапы 7, 9, 29, 31 (PG + MariaDB): UPDATE / DELETE + фильтры

**Критерии:**
- К любым 5-ти таблицам — запрос на изменение или удаление (+5 для каждой СУБД)
- К любым 3-ём таблицам — запрос с фильтром WHERE (+5 для каждой СУБД)

In [ ]:
# === PostgreSQL: UPDATE / DELETE ===

# 1. event_type: демонстрационный UPDATE (нормализация названия)
pg_cur.execute("UPDATE event_type SET event_type_name = 'listen' WHERE event_type_name = 'listen';")
pg_q('SELECT event_type_name AS "Тип события" FROM event_type ORDER BY event_type_id;',
     'event_type после UPDATE')

# 2. track: обнулить длину у трека с минимальным ID
pg_cur.execute('UPDATE track SET track_length_seconds = NULL WHERE track_id = (SELECT MIN(track_id) FROM track);')
print('track: длина обнулена у трека с MIN(track_id)')

# 3. album_track: удалить одну тестовую связь альбом-трек
pg_cur.execute('DELETE FROM album_track WHERE (album_id,track_id) = (SELECT album_id,track_id FROM album_track LIMIT 1);')
print('album_track: одна связь удалена')

# 4. artist_track: удалить одну тестовую связь артист-трек
pg_cur.execute('DELETE FROM artist_track WHERE (artist_id,track_id) = (SELECT artist_id,track_id FROM artist_track LIMIT 1);')
print('artist_track: одна связь удалена')

# 5. event: удалить самое раннее событие типа undislike
pg_cur.execute('DELETE FROM event WHERE event_id = (SELECT event_id FROM event WHERE event_type_id = 5 ORDER BY ts LIMIT 1);')
print('event: удалено одно самое раннее событие undislike')

In [ ]:
# === PostgreSQL: SELECT с фильтрами ===

# Фильтр 1: event — органические прослушивания >= 90% (3 FK -> 3 JOIN)
pg_q('''
    SELECT et.event_type_name                         AS "Тип события",
           ROUND(e.played_ratio_pct::numeric, 1)       AS "Прослушано (%)",
           t.track_length_seconds                      AS "Длина трека (сек)"
    FROM event e
    JOIN event_type et ON e.event_type_id = et.event_type_id
    JOIN track t       ON e.track_id      = t.track_id
    JOIN usr u         ON e.user_id       = u.user_id
    WHERE e.is_organic = 1
      AND e.played_ratio_pct >= 90
      AND et.event_type_name = \'listen\'
    LIMIT 10;
''', 'Фильтр: органические прослушивания >= 90%')

# Фильтр 2: track — треки длиннее 5 минут
pg_q('''
    SELECT track_length_seconds                   AS "Длина (сек)",
           ROUND(track_length_seconds/60.0, 2)    AS "Длина (мин)"
    FROM track
    WHERE track_length_seconds > 300
    ORDER BY track_length_seconds DESC
    LIMIT 10;
''', 'Фильтр: треки длиннее 5 минут')

# Фильтр 3: event_type — типы с \'like\' в названии
pg_q('''
    SELECT event_type_name AS "Тип события"
    FROM event_type
    WHERE event_type_name LIKE \'%like%\'
    ORDER BY event_type_id;
''', "Фильтр: типы событий с 'like' в названии")

In [ ]:
# === MariaDB: UPDATE / DELETE (те же 5 таблиц) ===

maria_cur.execute("UPDATE event_type SET event_type_name = 'listen' WHERE event_type_name = 'listen';")
maria_cur.execute('UPDATE track SET track_length_seconds = NULL WHERE track_id = (SELECT * FROM (SELECT MIN(track_id) FROM track) t);')
maria_cur.execute('DELETE FROM album_track  WHERE (album_id,track_id)  = (SELECT * FROM (SELECT album_id,track_id  FROM album_track  LIMIT 1) t);')
maria_cur.execute('DELETE FROM artist_track WHERE (artist_id,track_id) = (SELECT * FROM (SELECT artist_id,track_id FROM artist_track LIMIT 1) t);')
maria_cur.execute('DELETE FROM event WHERE event_id = (SELECT * FROM (SELECT event_id FROM event WHERE event_type_id = 5 ORDER BY ts LIMIT 1) t);')
maria_conn.commit()
print('MariaDB: UPDATE/DELETE выполнены')

In [ ]:
# === MariaDB: SELECT с фильтрами ===

maria_q('''
    SELECT et.event_type_name              AS `Тип события`,
           ROUND(e.played_ratio_pct, 1)    AS `Прослушано (%)`,
           t.track_length_seconds          AS `Длина трека (сек)`
    FROM event e
    JOIN event_type et ON e.event_type_id = et.event_type_id
    JOIN track t       ON e.track_id      = t.track_id
    JOIN usr u         ON e.user_id       = u.user_id
    WHERE e.is_organic = 1 AND e.played_ratio_pct >= 90
      AND et.event_type_name = \'listen\' LIMIT 10;
''', 'Фильтр MariaDB: органические прослушивания >= 90%')

maria_q('''
    SELECT track_length_seconds         AS `Длина (сек)`,
           ROUND(track_length_seconds/60, 2) AS `Длина (мин)`
    FROM track WHERE track_length_seconds > 300
    ORDER BY track_length_seconds DESC LIMIT 10;
''', 'Фильтр MariaDB: треки длиннее 5 минут')

maria_q("SELECT event_type_name AS `Тип события` FROM event_type WHERE event_type_name LIKE '%like%' ORDER BY event_type_id;",
        "Фильтр MariaDB: типы с 'like'")

---
## Этапы 9–12, 31–34 (PG + MariaDB): Хранимые процедуры

**Критерии:**
- К каждой таблице — 3 процедуры: `_insert`, `_update`, `_delete` (+4 за каждую СУБД)
- Тестирование 6 процедур для PostgreSQL (+4), 5 для MariaDB (+4)
- Продемонстрированы все процедуры (+1)
- Количество совпадает (+1)

**8 таблиц × 3 = 24 процедуры** на каждую СУБД.
Процедуры `_delete` блокируют удаление при наличии дочерних записей.

In [ ]:
PG_PROCS = '''
-- event_type
CREATE OR REPLACE PROCEDURE event_type_insert(p_name VARCHAR(20)) LANGUAGE plpgsql AS $$
BEGIN INSERT INTO event_type(event_type_name) VALUES(p_name) ON CONFLICT DO NOTHING;
EXCEPTION WHEN others THEN RAISE NOTICE \'event_type_insert: %\', SQLERRM; END; $$;

CREATE OR REPLACE PROCEDURE event_type_update(p_id INT, p_name VARCHAR(20)) LANGUAGE plpgsql AS $$
BEGIN UPDATE event_type SET event_type_name=p_name WHERE event_type_id=p_id;
EXCEPTION WHEN others THEN RAISE NOTICE \'event_type_update: %\', SQLERRM; END; $$;

CREATE OR REPLACE PROCEDURE event_type_delete(p_id INT) LANGUAGE plpgsql AS $$
DECLARE v INT; BEGIN SELECT COUNT(*) INTO v FROM event WHERE event_type_id=p_id;
IF v>0 THEN RAISE NOTICE \'event_type_delete: есть % событий, удаление невозможно\', v;
ELSE DELETE FROM event_type WHERE event_type_id=p_id; END IF; END; $$;

-- usr
CREATE OR REPLACE PROCEDURE usr_insert(p_id BIGINT) LANGUAGE plpgsql AS $$
BEGIN INSERT INTO usr(user_id) VALUES(p_id) ON CONFLICT DO NOTHING;
EXCEPTION WHEN others THEN RAISE NOTICE \'usr_insert: %\', SQLERRM; END; $$;

CREATE OR REPLACE PROCEDURE usr_update(p_old BIGINT, p_new BIGINT) LANGUAGE plpgsql AS $$
BEGIN UPDATE usr SET user_id=p_new WHERE user_id=p_old;
EXCEPTION WHEN others THEN RAISE NOTICE \'usr_update: %\', SQLERRM; END; $$;

CREATE OR REPLACE PROCEDURE usr_delete(p_id BIGINT) LANGUAGE plpgsql AS $$
DECLARE v INT; BEGIN SELECT COUNT(*) INTO v FROM event WHERE user_id=p_id;
IF v>0 THEN RAISE NOTICE \'usr_delete: есть % событий у пользователя\', v;
ELSE DELETE FROM usr WHERE user_id=p_id; END IF; END; $$;

-- artist
CREATE OR REPLACE PROCEDURE artist_insert(p_id INT) LANGUAGE plpgsql AS $$
BEGIN INSERT INTO artist(artist_id) VALUES(p_id) ON CONFLICT DO NOTHING;
EXCEPTION WHEN others THEN RAISE NOTICE \'artist_insert: %\', SQLERRM; END; $$;

CREATE OR REPLACE PROCEDURE artist_update(p_old INT, p_new INT) LANGUAGE plpgsql AS $$
BEGIN UPDATE artist SET artist_id=p_new WHERE artist_id=p_old;
EXCEPTION WHEN others THEN RAISE NOTICE \'artist_update: %\', SQLERRM; END; $$;

CREATE OR REPLACE PROCEDURE artist_delete(p_id INT) LANGUAGE plpgsql AS $$
DECLARE v INT; BEGIN SELECT COUNT(*) INTO v FROM artist_track WHERE artist_id=p_id;
IF v>0 THEN RAISE NOTICE \'artist_delete: есть % треков у артиста\', v;
ELSE DELETE FROM artist WHERE artist_id=p_id; END IF; END; $$;

-- album
CREATE OR REPLACE PROCEDURE album_insert(p_id INT) LANGUAGE plpgsql AS $$
BEGIN INSERT INTO album(album_id) VALUES(p_id) ON CONFLICT DO NOTHING;
EXCEPTION WHEN others THEN RAISE NOTICE \'album_insert: %\', SQLERRM; END; $$;

CREATE OR REPLACE PROCEDURE album_update(p_old INT, p_new INT) LANGUAGE plpgsql AS $$
BEGIN UPDATE album SET album_id=p_new WHERE album_id=p_old;
EXCEPTION WHEN others THEN RAISE NOTICE \'album_update: %\', SQLERRM; END; $$;

CREATE OR REPLACE PROCEDURE album_delete(p_id INT) LANGUAGE plpgsql AS $$
DECLARE v INT; BEGIN SELECT COUNT(*) INTO v FROM album_track WHERE album_id=p_id;
IF v>0 THEN RAISE NOTICE \'album_delete: есть % треков в альбоме\', v;
ELSE DELETE FROM album WHERE album_id=p_id; END IF; END; $$;

-- track
CREATE OR REPLACE PROCEDURE track_insert(p_id INT, p_len INT) LANGUAGE plpgsql AS $$
BEGIN INSERT INTO track(track_id,track_length_seconds) VALUES(p_id,p_len) ON CONFLICT DO NOTHING;
EXCEPTION WHEN others THEN RAISE NOTICE \'track_insert: %\', SQLERRM; END; $$;

CREATE OR REPLACE PROCEDURE track_update(p_id INT, p_len INT) LANGUAGE plpgsql AS $$
BEGIN UPDATE track SET track_length_seconds=p_len WHERE track_id=p_id;
EXCEPTION WHEN others THEN RAISE NOTICE \'track_update: %\', SQLERRM; END; $$;

CREATE OR REPLACE PROCEDURE track_delete(p_id INT) LANGUAGE plpgsql AS $$
DECLARE v INT; BEGIN SELECT COUNT(*) INTO v FROM event WHERE track_id=p_id;
IF v>0 THEN RAISE NOTICE \'track_delete: есть % событий для трека\', v;
ELSE DELETE FROM artist_track WHERE track_id=p_id;
     DELETE FROM album_track  WHERE track_id=p_id;
     DELETE FROM track WHERE track_id=p_id; END IF; END; $$;

-- artist_track
CREATE OR REPLACE PROCEDURE artist_track_insert(p_art INT, p_trk INT) LANGUAGE plpgsql AS $$
BEGIN INSERT INTO artist_track(artist_id,track_id) VALUES(p_art,p_trk) ON CONFLICT DO NOTHING;
EXCEPTION WHEN others THEN RAISE NOTICE \'artist_track_insert: %\', SQLERRM; END; $$;

CREATE OR REPLACE PROCEDURE artist_track_update(p_art_old INT, p_trk_old INT, p_art_new INT, p_trk_new INT)
LANGUAGE plpgsql AS $$
BEGIN DELETE FROM artist_track WHERE artist_id=p_art_old AND track_id=p_trk_old;
INSERT INTO artist_track(artist_id,track_id) VALUES(p_art_new,p_trk_new) ON CONFLICT DO NOTHING;
EXCEPTION WHEN others THEN RAISE NOTICE \'artist_track_update: %\', SQLERRM; END; $$;

CREATE OR REPLACE PROCEDURE artist_track_delete(p_art INT, p_trk INT) LANGUAGE plpgsql AS $$
BEGIN DELETE FROM artist_track WHERE artist_id=p_art AND track_id=p_trk;
EXCEPTION WHEN others THEN RAISE NOTICE \'artist_track_delete: %\', SQLERRM; END; $$;

-- album_track
CREATE OR REPLACE PROCEDURE album_track_insert(p_alb INT, p_trk INT) LANGUAGE plpgsql AS $$
BEGIN INSERT INTO album_track(album_id,track_id) VALUES(p_alb,p_trk) ON CONFLICT DO NOTHING;
EXCEPTION WHEN others THEN RAISE NOTICE \'album_track_insert: %\', SQLERRM; END; $$;

CREATE OR REPLACE PROCEDURE album_track_update(p_alb_old INT, p_trk_old INT, p_alb_new INT, p_trk_new INT)
LANGUAGE plpgsql AS $$
BEGIN DELETE FROM album_track WHERE album_id=p_alb_old AND track_id=p_trk_old;
INSERT INTO album_track(album_id,track_id) VALUES(p_alb_new,p_trk_new) ON CONFLICT DO NOTHING;
EXCEPTION WHEN others THEN RAISE NOTICE \'album_track_update: %\', SQLERRM; END; $$;

CREATE OR REPLACE PROCEDURE album_track_delete(p_alb INT, p_trk INT) LANGUAGE plpgsql AS $$
BEGIN DELETE FROM album_track WHERE album_id=p_alb AND track_id=p_trk;
EXCEPTION WHEN others THEN RAISE NOTICE \'album_track_delete: %\', SQLERRM; END; $$;

-- event
CREATE OR REPLACE PROCEDURE event_insert(p_uid BIGINT, p_tid INT, p_etid INT,
                                          p_ts BIGINT, p_org SMALLINT, p_ratio REAL)
LANGUAGE plpgsql AS $$
BEGIN INSERT INTO event(user_id,track_id,event_type_id,ts,is_organic,played_ratio_pct)
      VALUES(p_uid,p_tid,p_etid,p_ts,p_org,p_ratio);
EXCEPTION WHEN others THEN RAISE NOTICE \'event_insert: %\', SQLERRM; END; $$;

CREATE OR REPLACE PROCEDURE event_update(p_id BIGINT, p_ratio REAL) LANGUAGE plpgsql AS $$
BEGIN UPDATE event SET played_ratio_pct=p_ratio WHERE event_id=p_id;
EXCEPTION WHEN others THEN RAISE NOTICE \'event_update: %\', SQLERRM; END; $$;

CREATE OR REPLACE PROCEDURE event_delete(p_id BIGINT) LANGUAGE plpgsql AS $$
BEGIN DELETE FROM event WHERE event_id=p_id;
EXCEPTION WHEN others THEN RAISE NOTICE \'event_delete: %\', SQLERRM; END; $$;
'''

pg_cur.execute(PG_PROCS)
print('PostgreSQL: 24 хранимые процедуры созданы.')

# Проверка количества
pg_cur.execute("SELECT COUNT(*) FROM information_schema.routines WHERE routine_type='PROCEDURE' AND routine_schema='public';")
print('Процедур в public:', pg_cur.fetchone()[0])

In [ ]:
# === Тестирование 6 процедур (PostgreSQL) ===

# 1. Попытка вставить уже существующий тип события
print('Тест 1: event_type_insert (дубль — ожидаем NOTICE)')
pg_cur.execute("CALL event_type_insert('listen');")

# 2. Удаление типа события с дочерними записями
print('Тест 2: event_type_delete(1) — ожидаем блокировку')
pg_cur.execute('CALL event_type_delete(1);')

# 3. Вставка нового трека
print('Тест 3: track_insert(999999999, 180)')
pg_cur.execute('CALL track_insert(999999999, 180);')
pg_q('SELECT track_length_seconds AS "Длина" FROM track WHERE track_id = 999999999;',
     'Вставленный трек')

# 4. Обновление длины трека
print('Тест 4: track_update(999999999, 200)')
pg_cur.execute('CALL track_update(999999999, 200);')
pg_q('SELECT track_length_seconds AS "Длина" FROM track WHERE track_id = 999999999;',
     'Трек после UPDATE')

# 5. Вставка события
print('Тест 5: event_insert')
pg_cur.execute('''
    CALL event_insert(
        (SELECT user_id FROM usr LIMIT 1),
        999999999, 1, 1700000000, 1, 95.0
    );
''')
pg_q('''
    SELECT et.event_type_name AS "Тип", e.played_ratio_pct AS "Прослушано (%)"
    FROM event e
    JOIN event_type et ON e.event_type_id = et.event_type_id
    JOIN usr u         ON e.user_id       = u.user_id
    JOIN track t       ON e.track_id      = t.track_id
    WHERE t.track_id = 999999999 LIMIT 1;
''', 'Событие из теста')

# 6. Удаление трека при наличии событий — ожидаем блокировку
print('Тест 6: track_delete(999999999) — ожидаем NOTICE о блокировке')
pg_cur.execute('CALL track_delete(999999999);')

# Очистка тестовых данных
pg_cur.execute('DELETE FROM event WHERE track_id = 999999999;')
pg_cur.execute('DELETE FROM track WHERE track_id = 999999999;')
print('Тестовые данные удалены (трек 999999999 и его событие)')

### Хранимые процедуры MariaDB

В MariaDB процедуры создаются с `DELIMITER` и используют `BEGIN...END`.  
Синтаксис аналогичен PostgreSQL, но без `$$` и `LANGUAGE plpgsql`.

> **TBD:** Генерация 24 процедур MariaDB по тому же шаблону что и PostgreSQL.  
> Для pymysql `DELIMITER` не используется — процедуры передаются по одной через `execute()`.

In [ ]:
# === MariaDB: 24 хранимые процедуры (3 на каждую таблицу) ===

# Файл ma_procedures.sql содержит полный текст.
# Разделитель // обрабатывается вручную: split(' //') + execute по одному
_procs = r'''
DROP PROCEDURE IF EXISTS event_type_insert //
CREATE PROCEDURE event_type_insert(IN p_name VARCHAR(20))
BEGIN INSERT IGNORE INTO event_type(event_type_name) VALUES(p_name); END //

DROP PROCEDURE IF EXISTS event_type_update //
CREATE PROCEDURE event_type_update(IN p_id INT, IN p_name VARCHAR(20))
BEGIN UPDATE event_type SET event_type_name=p_name WHERE event_type_id=p_id; END //

DROP PROCEDURE IF EXISTS event_type_delete //
CREATE PROCEDURE event_type_delete(IN p_id INT)
BEGIN
    DECLARE v INT DEFAULT 0;
    SELECT COUNT(*) INTO v FROM event WHERE event_type_id = p_id;
    IF v > 0 THEN
        SIGNAL SQLSTATE '45000' SET MESSAGE_TEXT = 'Удаление невозможно: есть связанные события';
    ELSE
        DELETE FROM event_type WHERE event_type_id = p_id;
    END IF;
END //

DROP PROCEDURE IF EXISTS usr_insert //
CREATE PROCEDURE usr_insert(IN p_id BIGINT)
BEGIN INSERT IGNORE INTO usr(user_id) VALUES(p_id); END //

DROP PROCEDURE IF EXISTS usr_update //
CREATE PROCEDURE usr_update(IN p_old BIGINT, IN p_new BIGINT)
BEGIN UPDATE usr SET user_id=p_new WHERE user_id=p_old; END //

DROP PROCEDURE IF EXISTS usr_delete //
CREATE PROCEDURE usr_delete(IN p_id BIGINT)
BEGIN
    DECLARE v INT DEFAULT 0;
    SELECT COUNT(*) INTO v FROM event WHERE user_id = p_id;
    IF v > 0 THEN
        SIGNAL SQLSTATE '45000' SET MESSAGE_TEXT = 'Удаление невозможно: есть события у пользователя';
    ELSE
        DELETE FROM usr WHERE user_id = p_id;
    END IF;
END //

DROP PROCEDURE IF EXISTS artist_insert //
CREATE PROCEDURE artist_insert(IN p_id INT)
BEGIN INSERT IGNORE INTO artist(artist_id) VALUES(p_id); END //

DROP PROCEDURE IF EXISTS artist_update //
CREATE PROCEDURE artist_update(IN p_old INT, IN p_new INT)
BEGIN UPDATE artist SET artist_id=p_new WHERE artist_id=p_old; END //

DROP PROCEDURE IF EXISTS artist_delete //
CREATE PROCEDURE artist_delete(IN p_id INT)
BEGIN
    DECLARE v INT DEFAULT 0;
    SELECT COUNT(*) INTO v FROM artist_track WHERE artist_id = p_id;
    IF v > 0 THEN
        SIGNAL SQLSTATE '45000' SET MESSAGE_TEXT = 'Удаление невозможно: есть треки у артиста';
    ELSE
        DELETE FROM artist WHERE artist_id = p_id;
    END IF;
END //

DROP PROCEDURE IF EXISTS album_insert //
CREATE PROCEDURE album_insert(IN p_id INT)
BEGIN INSERT IGNORE INTO album(album_id) VALUES(p_id); END //

DROP PROCEDURE IF EXISTS album_update //
CREATE PROCEDURE album_update(IN p_old INT, IN p_new INT)
BEGIN UPDATE album SET album_id=p_new WHERE album_id=p_old; END //

DROP PROCEDURE IF EXISTS album_delete //
CREATE PROCEDURE album_delete(IN p_id INT)
BEGIN
    DECLARE v INT DEFAULT 0;
    SELECT COUNT(*) INTO v FROM album_track WHERE album_id = p_id;
    IF v > 0 THEN
        SIGNAL SQLSTATE '45000' SET MESSAGE_TEXT = 'Удаление невозможно: есть треки в альбоме';
    ELSE
        DELETE FROM album WHERE album_id = p_id;
    END IF;
END //

DROP PROCEDURE IF EXISTS track_insert //
CREATE PROCEDURE track_insert(IN p_id INT, IN p_len INT)
BEGIN INSERT IGNORE INTO track(track_id, track_length_seconds) VALUES(p_id, p_len); END //

DROP PROCEDURE IF EXISTS track_update //
CREATE PROCEDURE track_update(IN p_id INT, IN p_len INT)
BEGIN UPDATE track SET track_length_seconds=p_len WHERE track_id=p_id; END //

DROP PROCEDURE IF EXISTS track_delete //
CREATE PROCEDURE track_delete(IN p_id INT)
BEGIN
    DECLARE v INT DEFAULT 0;
    SELECT COUNT(*) INTO v FROM event WHERE track_id = p_id;
    IF v > 0 THEN
        SIGNAL SQLSTATE '45000' SET MESSAGE_TEXT = 'Удаление невозможно: есть события для трека';
    ELSE
        DELETE FROM artist_track WHERE track_id = p_id;
        DELETE FROM album_track  WHERE track_id = p_id;
        DELETE FROM track WHERE track_id = p_id;
    END IF;
END //

DROP PROCEDURE IF EXISTS artist_track_insert //
CREATE PROCEDURE artist_track_insert(IN p_art INT, IN p_trk INT)
BEGIN INSERT IGNORE INTO artist_track(artist_id, track_id) VALUES(p_art, p_trk); END //

DROP PROCEDURE IF EXISTS artist_track_update //
CREATE PROCEDURE artist_track_update(IN p_art_old INT, IN p_trk_old INT, IN p_art_new INT, IN p_trk_new INT)
BEGIN
    DELETE FROM artist_track WHERE artist_id=p_art_old AND track_id=p_trk_old;
    INSERT IGNORE INTO artist_track(artist_id, track_id) VALUES(p_art_new, p_trk_new);
END //

DROP PROCEDURE IF EXISTS artist_track_delete //
CREATE PROCEDURE artist_track_delete(IN p_art INT, IN p_trk INT)
BEGIN DELETE FROM artist_track WHERE artist_id=p_art AND track_id=p_trk; END //

DROP PROCEDURE IF EXISTS album_track_insert //
CREATE PROCEDURE album_track_insert(IN p_alb INT, IN p_trk INT)
BEGIN INSERT IGNORE INTO album_track(album_id, track_id) VALUES(p_alb, p_trk); END //

DROP PROCEDURE IF EXISTS album_track_update //
CREATE PROCEDURE album_track_update(IN p_alb_old INT, IN p_trk_old INT, IN p_alb_new INT, IN p_trk_new INT)
BEGIN
    DELETE FROM album_track WHERE album_id=p_alb_old AND track_id=p_trk_old;
    INSERT IGNORE INTO album_track(album_id, track_id) VALUES(p_alb_new, p_trk_new);
END //

DROP PROCEDURE IF EXISTS album_track_delete //
CREATE PROCEDURE album_track_delete(IN p_alb INT, IN p_trk INT)
BEGIN DELETE FROM album_track WHERE album_id=p_alb AND track_id=p_trk; END //

DROP PROCEDURE IF EXISTS event_insert //
CREATE PROCEDURE event_insert(IN p_uid BIGINT, IN p_tid INT, IN p_etid INT,
                               IN p_ts BIGINT, IN p_org TINYINT, IN p_ratio FLOAT)
BEGIN
    INSERT INTO event(user_id, track_id, event_type_id, ts, is_organic, played_ratio_pct)
    VALUES(p_uid, p_tid, p_etid, p_ts, p_org, p_ratio);
END //

DROP PROCEDURE IF EXISTS event_update //
CREATE PROCEDURE event_update(IN p_id BIGINT, IN p_ratio FLOAT)
BEGIN UPDATE event SET played_ratio_pct=p_ratio WHERE event_id=p_id; END //

DROP PROCEDURE IF EXISTS event_delete //
CREATE PROCEDURE event_delete(IN p_id BIGINT)
BEGIN DELETE FROM event WHERE event_id=p_id; END //
'''

# MariaDB: выполняем через exec_multi с DELIMITER //
for stmt in _procs.strip().split(' //'):
    stmt = stmt.strip()
    if stmt and not stmt.startswith('--'):
        try:
            maria_cur.execute(stmt)
        except Exception as e:
            print(f'  warn: {e}')
maria_conn.commit()

# Проверка количества процедур
maria_cur.execute("SELECT COUNT(*) FROM information_schema.routines WHERE routine_schema='yambda' AND routine_type='PROCEDURE'")
cnt = maria_cur.fetchone()[0]
print(f'MariaDB: создано {cnt} хранимых процедур')

---
## Этапы 13–15, 35–37 (PG + MariaDB): Виртуальные таблицы (VIEW)

**Критерии:**
- Виртуальные таблицы выявлены и реализованы верно (+3 за каждую СУБД)
- Продемонстрированы все реализованные VIEW (+1)
- Количество совпадает (+1)

**5 представлений**, отражающих бизнес-запросы к системе Yandex Music:

| VIEW | Описание |
|------|----------|
| `vw_track_full` | Трек + все артисты + все альбомы |
| `vw_event_stats` | Статистика событий по типу |
| `vw_top_tracks` | Топ треков по числу прослушиваний |
| `vw_user_activity` | Сводка активности пользователя |
| `vw_organic_listens` | Только органические прослушивания |


In [ ]:
PG_VIEWS = '''
-- 1. Полная информация о треке
CREATE OR REPLACE VIEW vw_track_full AS
    SELECT t.track_length_seconds                             AS "Длина (сек)",
           STRING_AGG(DISTINCT a.artist_id::text, \', \')    AS "Артисты",
           STRING_AGG(DISTINCT al.album_id::text, \', \')    AS "Альбомы"
    FROM track t
    LEFT JOIN artist_track att ON t.track_id  = att.track_id
    LEFT JOIN artist a         ON att.artist_id = a.artist_id
    LEFT JOIN album_track  abt ON t.track_id  = abt.track_id
    LEFT JOIN album al         ON abt.album_id  = al.album_id
    GROUP BY t.track_id, t.track_length_seconds;

-- 2. Статистика событий по типу
CREATE OR REPLACE VIEW vw_event_stats AS
    SELECT et.event_type_name                          AS "Тип события",
           COUNT(*)                                    AS "Количество",
           ROUND(AVG(e.played_ratio_pct)::numeric, 2)  AS "Среднее прослушивание (%)",
           SUM(CASE WHEN e.is_organic=1 THEN 1 ELSE 0 END) AS "Органических"
    FROM event e
    JOIN event_type et ON e.event_type_id = et.event_type_id
    JOIN usr u         ON e.user_id       = u.user_id
    JOIN track t       ON e.track_id      = t.track_id
    GROUP BY et.event_type_name;

-- 3. Топ треков по прослушиваниям
CREATE OR REPLACE VIEW vw_top_tracks AS
    SELECT COUNT(*)                              AS "Прослушиваний",
           t.track_length_seconds                AS "Длина (сек)",
           ROUND(AVG(e.played_ratio_pct)::numeric,1) AS "Среднее прослушивание (%)"
    FROM event e
    JOIN event_type et ON e.event_type_id = et.event_type_id
    JOIN usr u         ON e.user_id       = u.user_id
    JOIN track t       ON e.track_id      = t.track_id
    WHERE et.event_type_name = \'listen\'
    GROUP BY t.track_id, t.track_length_seconds
    ORDER BY COUNT(*) DESC;

-- 4. Сводка активности пользователей
CREATE OR REPLACE VIEW vw_user_activity AS
    SELECT COUNT(*)                                                        AS "Всего событий",
           SUM(CASE WHEN et.event_type_name=\'listen\'   THEN 1 ELSE 0 END) AS "Прослушиваний",
           SUM(CASE WHEN et.event_type_name=\'like\'     THEN 1 ELSE 0 END) AS "Лайков",
           SUM(CASE WHEN et.event_type_name=\'dislike\'  THEN 1 ELSE 0 END) AS "Дизлайков",
           ROUND(AVG(e.played_ratio_pct)::numeric, 1)                       AS "Среднее прослушивание (%)"
    FROM event e
    JOIN event_type et ON e.event_type_id = et.event_type_id
    JOIN usr u         ON e.user_id       = u.user_id
    JOIN track t       ON e.track_id      = t.track_id
    GROUP BY u.user_id;

-- 5. Органические прослушивания
CREATE OR REPLACE VIEW vw_organic_listens AS
    SELECT et.event_type_name                          AS "Тип события",
           ROUND(e.played_ratio_pct::numeric, 1)       AS "Прослушано (%)",
           t.track_length_seconds                      AS "Длина трека (сек)"
    FROM event e
    JOIN event_type et ON e.event_type_id = et.event_type_id
    JOIN usr u         ON e.user_id       = u.user_id
    JOIN track t       ON e.track_id      = t.track_id
    WHERE e.is_organic = 1;
'''

pg_cur.execute(PG_VIEWS)
print('PostgreSQL: 5 представлений созданы.')

pg_cur.execute("SELECT COUNT(*) FROM information_schema.tables WHERE table_schema='public' AND table_type='VIEW';")
print('VIEW в public:', pg_cur.fetchone()[0])

In [ ]:
pg_q('SELECT * FROM vw_event_stats ORDER BY "Количество" DESC;',
     'vw_event_stats')
pg_q('SELECT * FROM vw_top_tracks LIMIT 5;',
     'vw_top_tracks (топ-5)')
pg_q('SELECT * FROM vw_organic_listens LIMIT 5;',
     'vw_organic_listens (первые 5)')
pg_q('SELECT * FROM vw_track_full LIMIT 5;',
     'vw_track_full (первые 5)')
pg_q('SELECT * FROM vw_user_activity LIMIT 5;',
     'vw_user_activity (первые 5)')

In [ ]:
# === MariaDB: 5 представлений (VIEW) ===
# Адаптация PostgreSQL: STRING_AGG -> GROUP_CONCAT, ::numeric -> CAST AS DECIMAL

maria_cur.execute('DROP VIEW IF EXISTS vw_track_full')
maria_cur.execute('''
CREATE VIEW vw_track_full AS
    SELECT t.track_length_seconds                          AS `Длина (сек)`,
           GROUP_CONCAT(DISTINCT a.artist_id SEPARATOR ', ')  AS `Артисты`,
           GROUP_CONCAT(DISTINCT al.album_id SEPARATOR ', ')   AS `Альбомы`
    FROM track t
    LEFT JOIN artist_track att ON t.track_id  = att.track_id
    LEFT JOIN artist a         ON att.artist_id = a.artist_id
    LEFT JOIN album_track  abt ON t.track_id  = abt.track_id
    LEFT JOIN album al         ON abt.album_id  = al.album_id
    GROUP BY t.track_id, t.track_length_seconds
''')

maria_cur.execute('DROP VIEW IF EXISTS vw_event_stats')
maria_cur.execute('''
CREATE VIEW vw_event_stats AS
    SELECT et.event_type_name                           AS `Тип события`,
           COUNT(*)                                     AS `Количество`,
           ROUND(AVG(e.played_ratio_pct), 2)            AS `Среднее прослушивание (%)`,
           SUM(CASE WHEN e.is_organic=1 THEN 1 ELSE 0 END) AS `Органических`
    FROM event e
    JOIN event_type et ON e.event_type_id = et.event_type_id
    JOIN usr u         ON e.user_id       = u.user_id
    JOIN track t       ON e.track_id      = t.track_id
    GROUP BY et.event_type_name
''')

maria_cur.execute('DROP VIEW IF EXISTS vw_top_tracks')
maria_cur.execute('''
CREATE VIEW vw_top_tracks AS
    SELECT COUNT(*)                                AS `Прослушиваний`,
           t.track_length_seconds                  AS `Длина (сек)`,
           ROUND(AVG(e.played_ratio_pct), 1)        AS `Среднее прослушивание (%)`
    FROM event e
    JOIN event_type et ON e.event_type_id = et.event_type_id
    JOIN usr u         ON e.user_id       = u.user_id
    JOIN track t       ON e.track_id      = t.track_id
    WHERE et.event_type_name = 'listen'
    GROUP BY t.track_id, t.track_length_seconds
    ORDER BY COUNT(*) DESC
''')

maria_cur.execute('DROP VIEW IF EXISTS vw_user_activity')
maria_cur.execute('''
CREATE VIEW vw_user_activity AS
    SELECT COUNT(*)                                                    AS `Всего событий`,
           SUM(CASE WHEN et.event_type_name='listen'   THEN 1 ELSE 0 END) AS `Прослушиваний`,
           SUM(CASE WHEN et.event_type_name='like'     THEN 1 ELSE 0 END) AS `Лайков`,
           SUM(CASE WHEN et.event_type_name='dislike'  THEN 1 ELSE 0 END) AS `Дизлайков`,
           ROUND(AVG(e.played_ratio_pct), 1)                             AS `Среднее прослушивание (%)`
    FROM event e
    JOIN event_type et ON e.event_type_id = et.event_type_id
    JOIN usr u         ON e.user_id       = u.user_id
    JOIN track t       ON e.track_id      = t.track_id
    GROUP BY u.user_id
''')

maria_cur.execute('DROP VIEW IF EXISTS vw_organic_listens')
maria_cur.execute('''
CREATE VIEW vw_organic_listens AS
    SELECT et.event_type_name                           AS `Тип события`,
           ROUND(e.played_ratio_pct, 1)                 AS `Прослушано (%)`,
           t.track_length_seconds                      AS `Длина трека (сек)`
    FROM event e
    JOIN event_type et ON e.event_type_id = et.event_type_id
    JOIN usr u         ON e.user_id       = u.user_id
    JOIN track t       ON e.track_id      = t.track_id
    WHERE e.is_organic = 1
''')

maria_conn.commit()

maria_cur.execute("SELECT COUNT(*) FROM information_schema.tables WHERE table_schema='yambda' AND table_type='VIEW'")
print(f'MariaDB: создано {maria_cur.fetchone()[0]} представлений')

---
## Этапы 16–18, 38–41 (PG + MariaDB): Функции

**Критерии:**
- Реализованы и продемонстрированы все функции (+4 за каждую СУБД)
- Встроенные функции: без ограничений на количество (-1 за каждую пропущенную)
- Пользовательские функции: -1 за каждую пропущенную; кол-во совпадает (+1)

### Встроенные функции (в SELECT-запросах)
`COUNT`, `SUM`, `AVG`, `MAX`, `MIN`, `ROUND`, `STRING_AGG`, `CASE`, `LOWER`, `UPPER`,
`SUBSTRING`, `LENGTH`, `to_char`, `date_part`, `ROW_NUMBER() OVER`

### Пользовательские функции (2 штуки)
1. `fn_user_event_count(user_id)` — возвращает общее число событий пользователя
2. `fn_track_artist_list(track_id)` — возвращает строку с артистами трека

In [ ]:
# === Встроенные функции (PostgreSQL) ===

# COUNT, SUM, AVG, MAX, MIN, ROUND
pg_q('''
    SELECT COUNT(*)                                    AS "Всего событий",
           MAX(played_ratio_pct)                       AS "Макс. прослушивание (%)",
           MIN(played_ratio_pct)                       AS "Мин. прослушивание (%)",
           ROUND(AVG(played_ratio_pct)::numeric, 2)    AS "Среднее прослушивание (%)",
           SUM(CASE WHEN is_organic=1 THEN 1 ELSE 0 END) AS "Органических событий"
    FROM event;
''', 'Агрегаты по событиям')

# STRING_AGG, LOWER, UPPER, LENGTH
pg_q('''
    SELECT LOWER(event_type_name)  AS "Нижний регистр",
           UPPER(event_type_name)  AS "Верхний регистр",
           LENGTH(event_type_name) AS "Длина строки",
           SUBSTRING(event_type_name FROM 1 FOR 3) AS "Первые 3 символа"
    FROM event_type ORDER BY event_type_id;
''', 'Строковые функции')

# CASE, MAX, MIN на треках
pg_q('''
    SELECT MAX(track_length_seconds)                   AS "Самый длинный (сек)",
           MIN(track_length_seconds)                   AS "Самый короткий (сек)",
           CASE WHEN AVG(track_length_seconds) > 200
                THEN \'Длинные треки в среднем\'
                ELSE \'Короткие треки в среднем\' END  AS "Характеристика"
    FROM track WHERE track_length_seconds IS NOT NULL;
''', 'MIN/MAX/CASE по длине треков')

# ROW_NUMBER() OVER, to_char
pg_q('''
    SELECT ROW_NUMBER() OVER (ORDER BY COUNT(*) DESC) AS "Ранг",
           et.event_type_name                         AS "Тип события",
           COUNT(*)                                   AS "Количество событий"
    FROM event e
    JOIN event_type et ON e.event_type_id = et.event_type_id
    JOIN usr u         ON e.user_id       = u.user_id
    JOIN track t       ON e.track_id      = t.track_id
    GROUP BY et.event_type_name
    ORDER BY COUNT(*) DESC;
''', 'ROW_NUMBER() — рейтинг типов событий')

In [ ]:
# === Пользовательские функции (PostgreSQL) ===

pg_cur.execute('''
CREATE OR REPLACE FUNCTION fn_user_event_count(p_user_id BIGINT)
RETURNS BIGINT LANGUAGE plpgsql AS $$
DECLARE v_count BIGINT;
BEGIN
    SELECT COUNT(*) INTO v_count FROM event WHERE user_id = p_user_id;
    RETURN v_count;
END; $$;
''')

pg_cur.execute('''
CREATE OR REPLACE FUNCTION fn_track_artist_list(p_track_id INT)
RETURNS TEXT LANGUAGE plpgsql AS $$
DECLARE v_list TEXT;
BEGIN
    SELECT STRING_AGG(artist_id::text, \', \')
    INTO v_list
    FROM artist_track WHERE track_id = p_track_id;
    RETURN COALESCE(v_list, \'—\');
END; $$;
''')

print('PostgreSQL: 2 пользовательские функции созданы.')

# Подсчёт
pg_cur.execute("SELECT COUNT(*) FROM information_schema.routines WHERE routine_type='FUNCTION' AND routine_schema='public' AND routine_name LIKE 'fn\\_%';")
print('Пользовательских функций в public:', pg_cur.fetchone()[0])

# Демонстрация
pg_q('''
    SELECT fn_user_event_count(user_id) AS "Событий у пользователя"
    FROM usr LIMIT 5;
''', 'fn_user_event_count')

pg_q('''
    SELECT fn_track_artist_list(track_id) AS "Артисты трека"
    FROM track LIMIT 5;
''', 'fn_track_artist_list')

In [ ]:
# === MariaDB: встроенные функции ===

maria_q('''
    SELECT COUNT(*)                   AS `Всего событий`,
           MAX(played_ratio_pct)      AS `Макс. прослушивание (%)`,
           MIN(played_ratio_pct)      AS `Мин. прослушивание (%)`,
           ROUND(AVG(played_ratio_pct),2) AS `Среднее прослушивание (%)`,
           SUM(is_organic=1)          AS `Органических событий`
    FROM event;
''', 'Агрегаты MariaDB')

maria_q('''
    SELECT LOWER(event_type_name)              AS `Нижний регистр`,
           UPPER(event_type_name)              AS `Верхний регистр`,
           LENGTH(event_type_name)             AS `Длина строки`,
           SUBSTRING(event_type_name,1,3)      AS `Первые 3 символа`
    FROM event_type ORDER BY event_type_id;
''', 'Строковые функции MariaDB')

maria_q('''
    SELECT ROW_NUMBER() OVER (ORDER BY COUNT(*) DESC) AS `Ранг`,
           et.event_type_name                         AS `Тип события`,
           COUNT(*)                                   AS `Количество событий`
    FROM event e
    JOIN event_type et ON e.event_type_id = et.event_type_id
    JOIN usr u         ON e.user_id       = u.user_id
    JOIN track t       ON e.track_id      = t.track_id
    GROUP BY et.event_type_name ORDER BY COUNT(*) DESC;
''', 'ROW_NUMBER() MariaDB')

# === MariaDB: пользовательские функции (UDF) ===
# 1. Подсчёт событий пользователя
maria_cur.execute('DROP FUNCTION IF EXISTS fn_user_event_count')
maria_cur.execute('''
CREATE FUNCTION fn_user_event_count(p_user_id BIGINT)
RETURNS BIGINT READS SQL DATA
BEGIN
    DECLARE v_count BIGINT;
    SELECT COUNT(*) INTO v_count FROM event WHERE user_id = p_user_id;
    RETURN v_count;
END
''')

# 2. Список артистов трека
maria_cur.execute('DROP FUNCTION IF EXISTS fn_track_artist_list')
maria_cur.execute('''
CREATE FUNCTION fn_track_artist_list(p_track_id INT)
RETURNS TEXT READS SQL DATA
BEGIN
    DECLARE v_list TEXT;
    SELECT GROUP_CONCAT(artist_id SEPARATOR ', ')
    INTO v_list
    FROM artist_track WHERE track_id = p_track_id;
    RETURN COALESCE(v_list, '—');
END
''')

# 3. Косинусное сходство двух треков (1 = идентичны, 0 = ортогональны)
maria_cur.execute('DROP FUNCTION IF EXISTS fn_track_similarity')
maria_cur.execute('''
CREATE FUNCTION fn_track_similarity(p_track1 INT, p_track2 INT)
RETURNS DECIMAL(10,6) READS SQL DATA
BEGIN
    DECLARE v1 VECTOR(128);
    DECLARE v2 VECTOR(128);
    DECLARE dist DOUBLE;
    SELECT embed INTO v1 FROM track WHERE track_id = p_track1;
    SELECT embed INTO v2 FROM track WHERE track_id = p_track2;
    SET dist = VEC_Distance_Cosine(v1, v2);
    RETURN CAST(1.0 - dist AS DECIMAL(10,6));
END
''')

maria_conn.commit()

# Тест UDF
maria_q('SELECT fn_user_event_count((SELECT user_id FROM usr LIMIT 1)) AS `Событий пользователя`', 'UDF: user_event_count')
maria_q('SELECT fn_track_artist_list((SELECT track_id FROM artist_track LIMIT 1)) AS `Артисты трека`', 'UDF: track_artist_list')
maria_q('SELECT fn_track_similarity(1, 2) AS `Сходство треков 1 и 2`', 'UDF: track_similarity')

maria_cur.execute("SELECT COUNT(*) FROM information_schema.routines WHERE routine_type='FUNCTION' AND routine_schema='yambda'")
print(f"MariaDB UDF: создано {maria_cur.fetchone()[0]} функций")


---
## Бонус: Векторный поиск похожих треков

Демонстрация ключевого отличия этой реализации от стандартного школьного проекта —  
поиск похожих треков через эмбеддинги непосредственно в SQL.

**PostgreSQL** использует оператор `<=>` (косинусное расстояние, pgvector).  
**MariaDB** использует функцию `VEC_Distance_Cosine()` (MariaDB 11.7+).

Синтаксис подтверждён на реальном сервере; реализован в ячейке ниже и в файле `ma_vector_search.sql`.

In [ ]:
# === Поиск 5 треков, наиболее похожих на трек с минимальным ID (PostgreSQL) ===

pg_cur.execute('SELECT embed FROM track WHERE embed IS NOT NULL ORDER BY track_id LIMIT 1;')
query_vec = pg_cur.fetchone()[0]  # pgvector возвращает numpy array (register_vector)

pg_q('''
    SELECT t.track_length_seconds              AS "Длина (сек)",
           ROUND((t.embed <=> %s)::numeric, 4) AS "Косинусное расстояние"
    FROM track t
    WHERE t.embed IS NOT NULL
    ORDER BY t.embed <=> %s
    LIMIT 5;
''', 'Топ-5 похожих треков (pgvector)', params=(query_vec, query_vec))

In [ ]:
# === MariaDB: векторный поиск (VEC_Distance_Cosine) ===
# VECTOR INDEX IDX_track_embed (embed) DISTANCE=cosine

maria_q('''
    SELECT t.track_length_seconds                     AS `Длина (сек)`,
           ROUND(VEC_Distance_Cosine(
               t.embed,
               (SELECT embed FROM track ORDER BY track_id LIMIT 1)
           ), 4)                                     AS `Косинусное расстояние`
    FROM track t
    ORDER BY VEC_Distance_Cosine(
        t.embed,
        (SELECT embed FROM track ORDER BY track_id LIMIT 1)
    )
    LIMIT 5
''', 'MariaDB: 5 наиболее похожих треков')

# Тест функции сходства
maria_q('''
    SELECT fn_track_similarity(
        (SELECT track_id FROM track WHERE embed IS NOT NULL ORDER BY track_id LIMIT 1),
        (SELECT track_id FROM track WHERE embed IS NOT NULL ORDER BY track_id LIMIT 1 OFFSET 1)
    ) AS `Сходство треков`
''', 'MariaDB: косинусное сходство двух треков')

---
## ALTER DDL — демонстрация изменения структуры таблиц

Демонстрируются операции `ALTER TABLE` для обеих СУБД.  
Используется тестовая таблица, которая удаляется в конце.

In [ ]:
# === ALTER DDL (PostgreSQL) ===
pg_cur.execute('CREATE TABLE IF NOT EXISTS test_alter (id SERIAL PRIMARY KEY, name VARCHAR(40) NOT NULL, val INT NULL);')
pg_cur.execute('ALTER TABLE test_alter RENAME TO test_alter_renamed;')
pg_cur.execute('ALTER TABLE test_alter_renamed RENAME COLUMN name TO label;')
pg_cur.execute('ALTER TABLE test_alter_renamed ALTER COLUMN val TYPE DECIMAL(10,2);')
pg_cur.execute('ALTER TABLE test_alter_renamed ADD COLUMN created_at DATE NOT NULL DEFAULT CURRENT_DATE;')
pg_cur.execute('ALTER TABLE test_alter_renamed ADD CONSTRAINT CH_val CHECK (val > 0);')
pg_cur.execute('ALTER TABLE test_alter_renamed DROP CONSTRAINT CH_val;')
pg_cur.execute('ALTER TABLE test_alter_renamed ALTER COLUMN val SET DEFAULT 0;')
pg_cur.execute('ALTER TABLE test_alter_renamed ALTER COLUMN val DROP DEFAULT;')
pg_cur.execute('ALTER TABLE test_alter_renamed DROP COLUMN created_at;')
pg_cur.execute('DROP TABLE test_alter_renamed;')
print('PostgreSQL ALTER DDL: выполнено')

In [ ]:
# === ALTER DDL (MariaDB) ===
maria_cur.execute('CREATE TABLE IF NOT EXISTS test_alter (id INT NOT NULL AUTO_INCREMENT PRIMARY KEY, name VARCHAR(40) NOT NULL, val INT NULL);')
maria_cur.execute('ALTER TABLE test_alter RENAME TO test_alter_renamed;')
maria_cur.execute('ALTER TABLE test_alter_renamed CHANGE COLUMN name label VARCHAR(40) NOT NULL;')
maria_cur.execute('ALTER TABLE test_alter_renamed MODIFY COLUMN val DECIMAL(10,2);')
maria_cur.execute('ALTER TABLE test_alter_renamed ADD COLUMN created_at DATE NOT NULL DEFAULT (CURRENT_DATE);')
maria_cur.execute('ALTER TABLE test_alter_renamed ADD CONSTRAINT CH_val CHECK (val > 0);')
maria_cur.execute('ALTER TABLE test_alter_renamed DROP CONSTRAINT CH_val;')
maria_cur.execute('ALTER TABLE test_alter_renamed ALTER COLUMN val SET DEFAULT 0;')
maria_cur.execute('ALTER TABLE test_alter_renamed ALTER COLUMN val DROP DEFAULT;')
maria_cur.execute('ALTER TABLE test_alter_renamed DROP COLUMN created_at;')
maria_cur.execute('DROP TABLE test_alter_renamed;')
maria_conn.commit()
print('MariaDB ALTER DDL: выполнено')

---
## Итоговая сводка

Финальная проверка всех счётчиков для обеих СУБД.

In [ ]:
print('=== PostgreSQL ===')
checks = [
    ("Таблиц (BASE TABLE)",
     "SELECT COUNT(*) FROM information_schema.tables WHERE table_schema='public' AND table_type='BASE TABLE';"),
    ("Представлений (VIEW)",
     "SELECT COUNT(*) FROM information_schema.tables WHERE table_schema='public' AND table_type='VIEW';"),
    ("Процедур",
     "SELECT COUNT(*) FROM information_schema.routines WHERE routine_type='PROCEDURE' AND routine_schema='public';"),
    ("Функций",
     "SELECT COUNT(*) FROM information_schema.routines WHERE routine_type='FUNCTION' AND routine_schema='public';"),
]
for label, sql in checks:
    pg_cur.execute(sql)
    print(f'  {label}: {pg_cur.fetchone()[0]}')

print('\n=== Строк в таблицах (PostgreSQL) ===')
for tbl in ['event_type','usr','artist','album','track','artist_track','album_track','event']:
    pg_cur.execute(f'SELECT COUNT(*) FROM {tbl};')
    print(f'  {tbl}: {pg_cur.fetchone()[0]:,}')

print('\n=== MariaDB ===')
maria_cur.execute("SELECT COUNT(*) FROM information_schema.tables WHERE table_schema=DATABASE() AND table_type='BASE TABLE';")
print('  Таблиц:', maria_cur.fetchone()[0])
maria_cur.execute("SELECT COUNT(*) FROM information_schema.tables WHERE table_schema=DATABASE() AND table_type='VIEW';")
print('  Представлений:', maria_cur.fetchone()[0])
maria_cur.execute("SELECT COUNT(*) FROM information_schema.routines WHERE routine_schema=DATABASE() AND routine_type='PROCEDURE';")
print('  Процедур:', maria_cur.fetchone()[0])
maria_cur.execute("SELECT COUNT(*) FROM information_schema.routines WHERE routine_type='FUNCTION' AND routine_schema=DATABASE();")
print('  Функций:', maria_cur.fetchone()[0])

In [ ]:
# Закрытие соединений
pg_cur.close()
pg_conn.close()
maria_cur.close()
maria_conn.close()
print('Соединения закрыты.')

---
## Список TBD (статус выполнения)

| # | Пункт | Статус |
|---|-------|--------|
| 1 | Заполнить `PG_CONFIG` и `MARIADB_CONFIG` | ✅ Готово |
| 2 | MariaDB: загрузка треков с векторами (`VEC_FromText`) | ✅ 8 088 303 треков, 7 721 749 векторов |
| 3 | MariaDB: 24 хранимые процедуры | ✅ `ma_procedures.sql` |
| 4 | MariaDB: 5 представлений | ✅ `ma_views.sql` |
| 5 | MariaDB: 3 UDF (включая `fn_track_similarity`) | ✅ `ma_functions.sql` |
| 6 | MariaDB: векторный поиск `VEC_Distance_Cosine` | ✅ `ma_vector_search.sql` |
| 7 | Скриншоты pgAdmin и DBeaver | ⏳ Вручную |
| 8 | Оптимизация батч-загрузки | ✅ COPY + LOAD DATA, 12 чанков, parallel |
| 9 | Шаблон отчёта.docx | ✅ Сгенерирован |
| 10 | Финальная верификация `load_counts.sql` | ✅ Все 8 таблиц OK |